# PWKD Option 1 — Pretrained ResNet18 Student

Implements **Pruning While Knowledge Distillation** (Wang et al., 2025) using:
- **Teacher**: frozen pretrained ResNet50 (RadImageNet weights)
- **Student**: pretrained ResNet18 (RadImageNet weights), trained from a
  warm starting point rather than random initialisation

This is a cross-architecture setup. The student is smaller by design
(ResNet18 vs ResNet50), and PWKD simultaneously prunes it further and
distils knowledge from the teacher.

Produces 5 compressed models at pruning ratios [10%, 25%, 50%, 70%, 90%],
saved to `trained_models/pwkd_r18_<ratio>/` and uploaded to HuggingFace.

**Run all cells top to bottom. Requires GPU.**

In [1]:
import os, copy, subprocess
import torch

# navigate to project root
target = 'CS6423_knowledge_distillation_project'
if not os.getcwd().endswith(target):
    import sys
    os.chdir(os.path.join(os.getcwd(), target))
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

print(f'Working dir: {os.getcwd()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

subprocess.run(['pip', 'install', 'PyWavelets', '--quiet'], check=True)

Working dir: /home/cor10/CS6423_knowledge_distillation_project
Device: cuda


CompletedProcess(args=['pip', 'install', 'PyWavelets', '--quiet'], returncode=0)

In [2]:
import pandas as pd
from modules.dataset_prepper import datasetPrepper

data_prep = datasetPrepper(
    dataframe_path='data/labels.csv',
    image_dir='data/test_images',
).prepare(compute_class_weights=True)

NUM_CLASSES = len(data_prep.class_names)
print(f'Classes: {NUM_CLASSES}')
print(f'Train batches: {len(data_prep.train_loader)} | Val batches: {len(data_prep.val_loader)}')

Classes: 61
Train batches: 249 | Val batches: 50


In [3]:
from modules.imagenet_loader import ImagenetLoader
from modules.evaluate_model import ModelEvaluator

loader = ImagenetLoader()

# Teacher: frozen pretrained ResNet50
teacher = loader.load_radimagenet_resnet50(
    weights_path='trained_models/resnet50_baseline_gpu_new/resnet50_baseline_gpu_new.pth',
    load_type='load'
)
teacher = teacher.to(device).eval()
for p in teacher.parameters():
    p.requires_grad = False

evaluator = ModelEvaluator(
    data_loader=data_prep.val_loader,
    class_names=data_prep.class_names,
    device=str(device),
)

teacher_metrics = evaluator.evaluate_single(teacher, 'ResNet50_teacher')
print(f'Teacher F1: {teacher_metrics["f1_macro"]:.4f}')
print(f'Teacher params: {teacher_metrics["total_parameters"]:,}')


Warming up ResNet50_teacher...
Running inference...
Teacher F1: 0.4867
Teacher params: 23,633,021


In [4]:
# Student baseline: pretrained ResNet18 before any PWKD
# This gives us a warm starting point (F1 ~ 0.4) rather than random init
student_base = loader.load_radimagenet_resnet18(
    weights_path='trained_models/resnet18_baseline_gpu_new/resnet18_baseline_gpu_new.pth',
    load_type='load'
)
student_base = student_base.to(device)

baseline_metrics = evaluator.evaluate_single(student_base, 'ResNet18_pretrained_baseline')
BASELINE_PARAMS  = baseline_metrics['total_parameters']
print(f'ResNet18 pretrained baseline F1:    {baseline_metrics["f1_macro"]:.4f}')
print(f'ResNet18 pretrained baseline params: {BASELINE_PARAMS:,}')


Warming up ResNet18_pretrained_baseline...
Running inference...
ResNet18 pretrained baseline F1:    0.4054
ResNet18 pretrained baseline params: 11,207,805


## Channel count reference

ResNet18 and ResNet50 have different channel widths at each stage, so the
wavelet alignment modules need a 1×1 projection to match teacher → student:

| Stage   | ResNet50 (teacher) | ResNet18 (student) |
|---------|-------------------|-------------------|
| layer2  | 512               | 128               |
| layer3  | 1024              | 256               |
| layer4  | 2048              | 512               |

In [5]:
import torch.nn as nn
import copy
from modules.model_trainer import modelTrainer
from modules.evaluate_model import ModelEvaluator
# from pwkd import PWKDLoss, make_aux_fn, finalise_student
import importlib, pwkd
importlib.reload(pwkd)
from pwkd import PWKDLoss, make_aux_fn, finalise_student

TEACHER_CHANNELS = {'layer2': 512,  'layer3': 1024, 'layer4': 2048}
STUDENT_CHANNELS = {'layer2': 128,  'layer3': 256,  'layer4': 512}

PRUNING_RATIOS = [0.10, 0.25, 0.50, 0.70, 0.90]
NUM_EPOCHS     = 10
LAM            = 0.2
KD_TEMP        = 4.0
SPARSE_WEIGHT  = 1e-4

evaluator = ModelEvaluator(
    data_loader=data_prep.val_loader,
    class_names=data_prep.class_names,
    device=str(device),
)




pwkd_metrics = []

print(f'Teacher F1: {teacher_metrics["f1_macro"]:.4f}')
print(f'Teacher params: {teacher_metrics["total_parameters"]:,}')

print(f'ResNet18 pretrained baseline F1:    {baseline_metrics["f1_macro"]:.4f}')
print(f'ResNet18 pretrained baseline params: {BASELINE_PARAMS:,}')


for ratio in PRUNING_RATIOS:
    label = f'pwkd_r18_r{int(ratio * 100)}'
    print(f'\n{"="*60}')
    print(f'  PWKD Option 1 (ResNet18) — pruning ratio {ratio:.0%}')
    print(f'{"="*60}')

    # Each ratio starts from the same pretrained weights
    student = copy.deepcopy(student_base).to(device)
    for p in student.parameters():
        p.requires_grad = True

    pwkd_loss = PWKDLoss(
        student          = student,
        teacher          = teacher,
        pruning_ratio    = ratio,
        teacher_channels = TEACHER_CHANNELS,
        student_channels = STUDENT_CHANNELS,
        class_weights    = data_prep.class_weights.to(device)
                           if data_prep.class_weights is not None else None,
        lam              = LAM,
        kd_temp          = KD_TEMP,
        sparse_weight    = SPARSE_WEIGHT,
    ).to(device)

    trainer = modelTrainer(
        model      = student,
        data_prep  = data_prep,
        device     = device,
        learn_rate = 5e-4,
        num_epochs = NUM_EPOCHS,
        model_name = label,
    )
    trainer.loss_fn   = pwkd_loss
    trainer.optimizer = torch.optim.AdamW(
        list(student.parameters()) + list(pwkd_loss.parameters()),
        lr=5e-4, weight_decay=1e-2,
    )
    trainer.create_classnum_to_label_map(data_prep.class_names)
    trainer.train_all(save_as_object=True, aux_forward_fn=make_aux_fn(teacher), pwkd=True)

    student = finalise_student(student, pwkd_loss, data_prep.train_loader, device)

    # Save
    import os
    save_dir = os.path.join('trained_models', label)
    os.makedirs(save_dir, exist_ok=True)
    torch.save({'model': student, 'epoch': NUM_EPOCHS},
               os.path.join(save_dir, f'{label}_full.pth'))
    print(f'Saved → {save_dir}/{label}_full.pth')

    # Evaluate
    student.eval()
    metrics = evaluator.evaluate_single(student, label)

    pwkd_metrics.append({
        'Pruning Ratio':      f'{int(ratio*100)}%',
        'Size Reduction (%)': round((BASELINE_PARAMS - metrics['total_parameters'])
                                    / BASELINE_PARAMS * 100, 2),
        'F1 Score':           round(metrics['f1_macro'],     4),
        'Size (MB)':          round(metrics['model_size_mb'], 1),
        'Latency (ms)':       round(metrics['avg_latency_ms'], 2),
    })
    print(f'  ratio {ratio:.0%} | F1: {metrics["f1_macro"]:.4f} | '
          f'params: {metrics["total_parameters"]:,} | '
          f'latency: {metrics["avg_latency_ms"]:.2f}ms')

summary_df = pd.DataFrame(pwkd_metrics).set_index('Pruning Ratio')
print('\nPWKD Option 1 (ResNet18) — Results')
display(summary_df)


Teacher F1: 0.4867
Teacher params: 23,633,021
ResNet18 pretrained baseline F1:    0.4054
ResNet18 pretrained baseline params: 11,207,805

  PWKD Option 1 (ResNet18) — pruning ratio 10%
  Epoch 1/10 [  4.8%]  loss: 2.4408
  Epoch 1/10 [  9.6%]  loss: 2.8545
  Epoch 1/10 [ 14.5%]  loss: 2.3419
  Epoch 1/10 [ 19.3%]  loss: 2.1466
  Epoch 1/10 [ 24.1%]  loss: 2.0009
  Epoch 1/10 [ 28.9%]  loss: 1.8433
  Epoch 1/10 [ 33.7%]  loss: 1.9103
  Epoch 1/10 [ 38.6%]  loss: 1.7448
  Epoch 1/10 [ 43.4%]  loss: 1.3310
  Epoch 1/10 [ 48.2%]  loss: 1.4084
  Epoch 1/10 [ 53.0%]  loss: 1.3874
  Epoch 1/10 [ 57.8%]  loss: 1.2904
  Epoch 1/10 [ 62.7%]  loss: 1.1842
  Epoch 1/10 [ 67.5%]  loss: 1.3218
  Epoch 1/10 [ 72.3%]  loss: 1.1949
  Epoch 1/10 [ 77.1%]  loss: 1.2818
  Epoch 1/10 [ 81.9%]  loss: 1.1967
  Epoch 1/10 [ 86.7%]  loss: 1.1878
  Epoch 1/10 [ 91.6%]  loss: 1.0461
  Epoch 1/10 [ 96.4%]  loss: 1.0707
  Epoch 1/10 [100.0%]  loss: 1.0757


Validating: 100%|██████████| 50/50 [00:02<00:00, 18.27batch/s]


Epoch 1/10
Train Loss: 1.5899 | Train F1: 0.4362
Val Loss: 2.1114 | Val F1: 0.2586
Epoch Time: 38.05s



  Epoch 2/10 [  4.8%]  loss: 0.9068
  Epoch 2/10 [  9.6%]  loss: 1.1005
  Epoch 2/10 [ 14.5%]  loss: 1.0496
  Epoch 2/10 [ 19.3%]  loss: 0.9510
  Epoch 2/10 [ 24.1%]  loss: 0.9796
  Epoch 2/10 [ 28.9%]  loss: 0.9695
  Epoch 2/10 [ 33.7%]  loss: 0.9246
  Epoch 2/10 [ 38.6%]  loss: 0.8854
  Epoch 2/10 [ 43.4%]  loss: 0.8808
  Epoch 2/10 [ 48.2%]  loss: 0.8564
  Epoch 2/10 [ 53.0%]  loss: 0.8375
  Epoch 2/10 [ 57.8%]  loss: 0.8559
  Epoch 2/10 [ 62.7%]  loss: 0.8140
  Epoch 2/10 [ 67.5%]  loss: 0.8068
  Epoch 2/10 [ 72.3%]  loss: 0.8375
  Epoch 2/10 [ 77.1%]  loss: 0.7806
  Epoch 2/10 [ 81.9%]  loss: 0.7978
  Epoch 2/10 [ 86.7%]  loss: 0.8493
  Epoch 2/10 [ 91.6%]  loss: 0.7624
  Epoch 2/10 [ 96.4%]  loss: 0.7301
  Epoch 2/10 [100.0%]  loss: 0.7060


Validating: 100%|██████████| 50/50 [00:02<00:00, 18.16batch/s]

...

  Epoch 10/10 [  4.8%]  loss: 0.4955
  Epoch 10/10 [  9.6%]  loss: 0.5397
  Epoch 10/10 [ 14.5%]  loss: 0.4782
  Epoch 10/10 [ 19.3%]  loss: 0.5588
  Epoch 10/10 [ 24.1%]  loss: 0.5066
  Epoch 10/10 [ 28.9%]  loss: 0.5193
  Epoch 10/10 [ 33.7%]  loss: 0.6003
  Epoch 10/10 [ 38.6%]  loss: 0.5262
  Epoch 10/10 [ 43.4%]  loss: 0.5025
  Epoch 10/10 [ 48.2%]  loss: 0.5145
  Epoch 10/10 [ 53.0%]  loss: 0.4921
  Epoch 10/10 [ 57.8%]  loss: 0.4729
  Epoch 10/10 [ 62.7%]  loss: 0.4936
  Epoch 10/10 [ 67.5%]  loss: 0.4694
  Epoch 10/10 [ 72.3%]  loss: 0.4475
  Epoch 10/10 [ 77.1%]  loss: 0.4740
  Epoch 10/10 [ 81.9%]  loss: 0.4718
  Epoch 10/10 [ 86.7%]  loss: 0.4856
  Epoch 10/10 [ 91.6%]  loss: 0.4576
  Epoch 10/10 [ 96.4%]  loss: 0.4750
  Epoch 10/10 [100.0%]  loss: 0.4866


Validating: 100%|██████████| 50/50 [00:02<00:00, 18.10batch/s]


Epoch 10/10
Train Loss: 0.4986 | Train F1: 0.7687
Val Loss: 1.8981 | Val F1: 0.3407
Epoch Time: 38.30s



Finalised: 1724/1920 conv1 channels zeroed (89.8%)
Saved → trained_models/pwkd_r18_r90/pwkd_r18_r90_full.pth

Warming up pwkd_r18_r90...
Running inference...
  ratio 90% | F1: 0.0001 | params: 1,339,197 | latency: 0.39ms

PWKD Option 1 (ResNet18) — Results


,Size Reduction (%),F1 Score,Size (MB),Latency (ms)
Pruning Ratio,,,,
10%,9.69,0.3370,42.8,0.38
25%,24.50,0.2729,42.8,0.40
50%,49.01,0.0009,42.8,0.39
70%,68.50,0.0001,42.8,0.38
90%,88.05,0.0001,42.8,0.39


In [5]:
# only run this cell if not running the previous cell

import torch.nn as nn
import copy
from modules.model_trainer import modelTrainer
from modules.evaluate_model import ModelEvaluator
# from pwkd import PWKDLoss, make_aux_fn, finalise_student
import importlib, pwkd
importlib.reload(pwkd)
from pwkd import PWKDLoss, make_aux_fn, finalise_student

TEACHER_CHANNELS = {'layer2': 512,  'layer3': 1024, 'layer4': 2048}
STUDENT_CHANNELS = {'layer2': 128,  'layer3': 256,  'layer4': 512}

PRUNING_RATIOS = [0.10, 0.25, 0.50, 0.70, 0.90]
NUM_EPOCHS     = 10
LAM            = 0.2
KD_TEMP        = 4.0
SPARSE_WEIGHT  = 1e-4

evaluator = ModelEvaluator(
    data_loader=data_prep.val_loader,
    class_names=data_prep.class_names,
    device=str(device),
)




pwkd_metrics = []

In [6]:
# ── Extra ratios to better capture the Pareto curve descent ──────────────────
# Run this cell independently — existing trained models are not affected.

EXTRA_RATIOS = [0.2, 0.3, 0.35, 0.40, 0.45]   # between the good and collapsed points

for ratio in EXTRA_RATIOS:
    label = f'pwkd_r18_r{int(ratio * 100)}'
    print(f'\n{"="*60}')
    print(f'  PWKD Option 1 (ResNet18) — pruning ratio {ratio:.0%}')
    print(f'{"="*60}')

    student = copy.deepcopy(student_base).to(device)
    for p in student.parameters():
        p.requires_grad = True

    pwkd_loss = PWKDLoss(
        student          = student,
        teacher          = teacher,
        pruning_ratio    = ratio,
        teacher_channels = TEACHER_CHANNELS,
        student_channels = STUDENT_CHANNELS,
        class_weights    = data_prep.class_weights.to(device)
                           if data_prep.class_weights is not None else None,
        lam              = LAM,
        kd_temp          = KD_TEMP,
        sparse_weight    = SPARSE_WEIGHT,
    ).to(device)

    trainer = modelTrainer(
        model      = student,
        data_prep  = data_prep,
        device     = device,
        learn_rate = 5e-4,
        num_epochs = NUM_EPOCHS,
        model_name = label,
    )
    trainer.loss_fn   = pwkd_loss
    trainer.optimizer = torch.optim.AdamW(
        list(student.parameters()) + list(pwkd_loss.parameters()),
        lr=5e-4, weight_decay=1e-2,
    )
    trainer.create_classnum_to_label_map(data_prep.class_names)
    trainer.train_all(save_as_object=True, aux_forward_fn=make_aux_fn(teacher), pwkd=True)

    student = finalise_student(student, pwkd_loss, data_prep.train_loader, device)

    save_dir = os.path.join('trained_models', label)
    os.makedirs(save_dir, exist_ok=True)
    torch.save({'model': student, 'epoch': NUM_EPOCHS},
               os.path.join(save_dir, f'{label}_full.pth'))
    print(f'Saved → {save_dir}/{label}_full.pth')

    student.eval()
    metrics = evaluator.evaluate_single(student, label)

    pwkd_metrics.append({
        'Pruning Ratio':      f'{int(ratio*100)}%',
        'Size Reduction (%)': round((BASELINE_PARAMS - metrics['total_parameters'])
                                    / BASELINE_PARAMS * 100, 2),
        'F1 Score':           round(metrics['f1_macro'],     4),
        'Size (MB)':          round(metrics['model_size_mb'], 1),
        'Latency (ms)':       round(metrics['avg_latency_ms'], 2),
    })
    print(f'  ratio {ratio:.0%} | F1: {metrics["f1_macro"]:.4f} | '
          f'params: {metrics["total_parameters"]:,}')

summary_df = pd.DataFrame(pwkd_metrics).set_index('Pruning Ratio')
display(summary_df)


  PWKD Option 1 (ResNet18) — pruning ratio 20%
  Epoch 1/10 [  4.8%]  loss: 2.6207
  Epoch 1/10 [  9.6%]  loss: 2.8309
  Epoch 1/10 [ 14.5%]  loss: 2.4174
  Epoch 1/10 [ 19.3%]  loss: 1.9418
  Epoch 1/10 [ 24.1%]  loss: 1.9137
  Epoch 1/10 [ 28.9%]  loss: 1.9062
  Epoch 1/10 [ 33.7%]  loss: 1.8461
  Epoch 1/10 [ 38.6%]  loss: 1.6924
  Epoch 1/10 [ 43.4%]  loss: 1.6959
  Epoch 1/10 [ 48.2%]  loss: 1.6498
  Epoch 1/10 [ 53.0%]  loss: 1.5759
  Epoch 1/10 [ 57.8%]  loss: 1.4958
  Epoch 1/10 [ 62.7%]  loss: 1.4504
  Epoch 1/10 [ 67.5%]  loss: 1.3912
  Epoch 1/10 [ 72.3%]  loss: 1.2050
  Epoch 1/10 [ 77.1%]  loss: 1.2093
  Epoch 1/10 [ 81.9%]  loss: 1.2671
  Epoch 1/10 [ 86.7%]  loss: 1.1061
  Epoch 1/10 [ 91.6%]  loss: 1.1098
  Epoch 1/10 [ 96.4%]  loss: 1.1329
  Epoch 1/10 [100.0%]  loss: 0.9995


Validating: 100%|██████████| 50/50 [00:02<00:00, 18.16batch/s]


Epoch 1/10
Train Loss: 1.6486 | Train F1: 0.4167
Val Loss: 2.0520 | Val F1: 0.2792
Epoch Time: 38.14s



  Epoch 2/10 [  4.8%]  loss: 1.1277
  Epoch 2/10 [  9.6%]  loss: 1.0431
  Epoch 2/10 [ 14.5%]  loss: 0.9882
  Epoch 2/10 [ 19.3%]  loss: 1.0144
  Epoch 2/10 [ 24.1%]  loss: 0.9416
  Epoch 2/10 [ 28.9%]  loss: 0.9811
  Epoch 2/10 [ 33.7%]  loss: 0.9259
  Epoch 2/10 [ 38.6%]  loss: 0.9126
  Epoch 2/10 [ 43.4%]  loss: 0.9139
  Epoch 2/10 [ 48.2%]  loss: 0.8244
  Epoch 2/10 [ 53.0%]  loss: 0.7961
  Epoch 2/10 [ 57.8%]  loss: 0.8396
  Epoch 2/10 [ 62.7%]  loss: 0.8440
  Epoch 2/10 [ 67.5%]  loss: 0.8826
  Epoch 2/10 [ 72.3%]  loss: 0.8007
  Epoch 2/10 [ 77.1%]  loss: 0.9049
  Epoch 2/10 [ 81.9%]  loss: 0.8109
  Epoch 2/10 [ 86.7%]  loss: 0.8543
  Epoch 2/10 [ 91.6%]  loss: 0.9753
  Epoch 2/10 [ 96.4%]  loss: 1.0217
  Epoch 2/10 [100.0%]  loss: 1.1500


Validating: 100%|██████████| 50/50 [00:02<00:00, 18.23batch/s]

...

  Epoch 10/10 [  4.8%]  loss: 0.4532
  Epoch 10/10 [  9.6%]  loss: 0.4917
  Epoch 10/10 [ 14.5%]  loss: 0.4347
  Epoch 10/10 [ 19.3%]  loss: 0.4296
  Epoch 10/10 [ 24.1%]  loss: 0.4469
  Epoch 10/10 [ 28.9%]  loss: 0.4400
  Epoch 10/10 [ 33.7%]  loss: 0.4384
  Epoch 10/10 [ 38.6%]  loss: 0.4599
  Epoch 10/10 [ 43.4%]  loss: 0.4229
  Epoch 10/10 [ 48.2%]  loss: 0.4725
  Epoch 10/10 [ 53.0%]  loss: 0.4589
  Epoch 10/10 [ 57.8%]  loss: 0.4721
  Epoch 10/10 [ 62.7%]  loss: 0.4654
  Epoch 10/10 [ 67.5%]  loss: 0.4787
  Epoch 10/10 [ 72.3%]  loss: 0.4614
  Epoch 10/10 [ 77.1%]  loss: 0.4380
  Epoch 10/10 [ 81.9%]  loss: 0.5069
  Epoch 10/10 [ 86.7%]  loss: 0.4380
  Epoch 10/10 [ 91.6%]  loss: 0.4683
  Epoch 10/10 [ 96.4%]  loss: 0.4614
  Epoch 10/10 [100.0%]  loss: 0.4608


Validating: 100%|██████████| 50/50 [00:02<00:00, 18.18batch/s]


Epoch 10/10
Train Loss: 0.4571 | Train F1: 0.7851
Val Loss: 1.9155 | Val F1: 0.3245
Epoch Time: 38.27s



Finalised: 860/1920 conv1 channels zeroed (44.8%)
Saved → trained_models/pwkd_r18_r45/pwkd_r18_r45_full.pth

Warming up pwkd_r18_r45...
Running inference...
  ratio 45% | F1: 0.0055 | params: 6,276,669


,Size Reduction (%),F1 Score,Size (MB),Latency (ms)
Pruning Ratio,,,,
20%,19.49,0.2680,42.8,0.39
30%,29.24,0.1004,42.8,0.39
35%,34.20,0.0352,42.8,0.39
40%,39.04,0.0124,42.8,0.39
45%,44.00,0.0055,42.8,0.39
